In [1]:
!git clone https://huggingface.co/ctheodoris/Geneformer; cd Geneformer; pip install .

Cloning into 'Geneformer'...
remote: Enumerating objects: 1082, done.
remote: Counting objects: 100% (382/382), done.
remote: Compressing objects: 100% (381/381), done.
remote: Total 1082 (delta 229), reused 0 (delta 0), pack-reused 700 (from 1)
Receiving objects: 100% (1082/1082), 5.73 MiB | 5.41 MiB/s, done.
Resolving deltas: 100% (656/656), done.
Filtering content: 100% (27/27), 1.47 GiB | 33.71 MiB/s, done.
Processing /content/Geneformer
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 k

In [2]:
!pip install anndata scanpy transformers datasets torch h5py cellxgene_census mygene

INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.9/78.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 60.3 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
import scanpy as sc
import os
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
import random
import re
import os
import scipy.sparse
from anndata import AnnData
import gc
import h5py
import scipy.sparse as sparse
from scipy.sparse import csr_matrix, vstack
import anndata as ad
import anndata
import torch
from datasets import Dataset, load_from_disk
from geneformer import TranscriptomeTokenizer
import mygene

In [5]:



def inspect_h5ad(dataset):

    file_path = f"{dataset}"  # Update with your dataset path

    try:
        with h5py.File(file_path, "r") as f:
            print(f"\nInspecting {file_path}...")

            # Check if 'X' exists and is a dataset
            shape = "Unknown"
            if "X" in f:
                if isinstance(f["X"], h5py.Dataset):
                    shape = f["X"].shape
                else:
                    print("Warning: 'X' is a group, not a dataset.")

            print("Shape (cells x genes):", shape)

            # Observations (obs metadata)
            obs_keys = list(f["obs"].keys()) if "obs" in f else []
            print("\n--- Observations (adata.obs) ---")
            print(f"Metadata columns in obs: {obs_keys}")

            # Variables (var metadata)
            var_keys = list(f["var"].keys()) if "var" in f else []
            print("\n--- Variables (adata.var) ---")
            print(f"Metadata columns in var: {var_keys}")

            # Layers (additional data matrices)
            layers = list(f["layers"].keys()) if "layers" in f else []
            print("\n--- Layers ---")
            print(f"Available layers: {layers}")

            # Embeddings (obsm)
            obsm_keys = list(f["obsm"].keys()) if "obsm" in f else []
            print("\n--- Embeddings (obsm) ---")
            print(f"Available embeddings: {obsm_keys}")

            # Unstructured data (uns)
            uns_keys = list(f["uns"].keys()) if "uns" in f else []
            print("\n--- Unstructured Data (uns) ---")
            print(f"Keys in uns: {uns_keys}")

            #Extracting var names from the index.



            print("-" * 100)

    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
    except Exception as e:
        print(f"An unexpected error occurred while reading {file_path}: {e}")




In [14]:
inspect_h5ad("/content/drive/MyDrive/cell2text_dataset/batch_98000.h5ad" )


Inspecting /content/drive/MyDrive/cell2text_dataset/batch_98000.h5ad...
Shape (cells x genes): Unknown

--- Observations (adata.obs) ---
Metadata columns in obs: ['_index', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'disease', 'disease_ontology_term_id', 'n_counts', 'sex', 'soma_joinid', 'tissue_general', 'tissue_general_ontology_term_id']

--- Variables (adata.var) ---
Metadata columns in var: ['_index', 'ensembl_id', 'feature_id', 'feature_length', 'feature_name', 'feature_type', 'n_measured_obs', 'nnz', 'soma_joinid']

--- Layers ---
Available layers: []

--- Embeddings (obsm) ---
Available embeddings: []

--- Unstructured Data (uns) ---
Keys in uns: []
----------------------------------------------------------------------------------------------------


In [44]:
# Parse the metadata text from the ontologies
def parse_cell_ontology(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    def extract_info(cell_id, entry):
        """ Extracts relevant fields while handling nested structures. """
        return {
            "id": cell_id,  # Extract the outer dictionary key as the ID
            "name": entry.get("name", ""),
            "definition": entry.get("def", ""),
            "synonyms": entry.get("synonym", []) if isinstance(entry.get("synonym"), list) else [entry.get("synonym")] if entry.get("synonym") else [],
        }

    parsed_data = [extract_info(cell_id, details) for cell_id, details in data.items()]

    return parsed_data

In [45]:
def tokenize_with_geneformer(input_dir, output_dir, dataset_name):
    """
    Tokenize an h5ad file using Geneformer's TranscriptomeTokenizer and save as an Arrow file.

    Parameters:
    -----------
    h5ad_path : str
        Path to the input .h5ad file.
    output_dir : str
        Directory to save the tokenized dataset.
    dataset_name : str
        Name of the dataset.

    Returns:
    --------
    str
        Path to the saved tokenized dataset.
    """
    # Default custom attributes
    custom_attrs = {
        "soma_joinid": "soma_joinid",
        "cell_type": "cell_type",
        "cell_type_ontology_term_id": "cell_type_ontology_term_id",
        "development_stage": "development_stage",
        "disease": "disease",
        "disease_ontology_term_id": "disease_ontology_term_id",
        "sex": "sex",
        "tissue_general": "tissue_general",
        "tissue_general_ontology_term_id": "tissue_general_ontology_term_id",
    }

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Initialize Tokenizer
    tk = TranscriptomeTokenizer(custom_attrs)

    # Tokenize the data
    tk.tokenize_data(
        input_dir,
        output_dir,
        dataset_name,
        file_format="h5ad"
    )


In [46]:
def add_struct_description(dataset: Dataset, ontology_dict: dict, batch_size: int = 1000) -> Dataset:
    """Add a structured text description column to a Hugging Face dataset based on ontology info and top genes."""

    # Identify all ontology columns
    ontology_columns = [col for col in dataset.column_names if col.endswith('_ontology_term_id')]
    base_columns = {col: col.replace('_ontology_term_id', '') for col in ontology_columns}

    # Function to process batches
    def process_batch(examples):
        batch_descriptions = []
        batch_size_actual = len(examples[list(examples.keys())[0]])  # Get actual batch size

        for i in range(batch_size_actual):
            cell_desc_parts = []

            # Get values for this example
            example = {key: values[i] for key, values in examples.items()}

            # Handle ontology-based description
            for onto_col, base_col in base_columns.items():
                ontology_id = str(example.get(onto_col, '')).upper() if example.get(onto_col) else ""
                base_value = example.get(base_col, '')

                if not ontology_id:
                    continue

                definition = ""
                if ontology_id in ontology_dict:
                    definition = ontology_dict[ontology_id].get('definition', '')

                if base_value and definition:
                    desc_part = f"{base_col.replace('_', ' ').title()}: {base_value}, {definition}"
                    cell_desc_parts.append(desc_part)
                elif base_value:
                    desc_part = f"{base_col.replace('_', ' ').title()}: {base_value}"
                    cell_desc_parts.append(desc_part)


            # For age
            age = example.get("development_stage", "")
            if age and age.lower() != 'unknown':
                cell_desc_parts.append(f"Development Stage: {age}")

            # For sex
            sex = example.get("sex", "")
            if sex and sex.lower() != 'unknown':
                cell_desc_parts.append(f"Sex: {sex}")

            # Construct the final structured description
            struct_desc = "; ".join(cell_desc_parts) + "." if cell_desc_parts else ""
            batch_descriptions.append(struct_desc)

        examples["struct_desc"] = batch_descriptions
        return examples

    # Apply function to entire dataset with explicit batching
    print("Adding structured descriptions to all examples...")
    return dataset.map(
        process_batch,
        batched=True,
        batch_size=batch_size,
        desc="Adding structured descriptions"
    )

In [47]:
def process_dataset(
    input_dir="/content/drive/MyDrive/cell2text_dataset",
    output_dir="/content/drive/MyDrive/cell2text_dataset_final",
    dataset_name="final_dataset",
    ontology_filepath="/content/drive/MyDrive/obo.json",
    batch_size=1000  # Add batch_size parameter
):
    print("Tokenizing dataset...")

    #Tokenize the dataset
    tokenize_with_geneformer(input_dir, output_dir, dataset_name)

    print("Loading tokenized dataset...")
    # Load the tokenized dataset
    hf_dataset = load_from_disk("/content/drive/MyDrive/cell2text_dataset_final/final_dataset.dataset")

    print(f"Dataset info: {hf_dataset}")
    print(f"Columns: {hf_dataset.column_names}")
    print(f"Shape: {hf_dataset.shape}")
    print(f"Total number of examples: {len(hf_dataset)}")



    print("Adding structured descriptions...")
    #Load ontology dictionary
    parsed_cells = parse_cell_ontology(ontology_filepath)
    ontology_dict = {
        cell["id"]: {
            "name": cell["name"],
            "definition": cell["definition"],
            "synonym": cell["synonyms"]
        }
        for cell in parsed_cells
    }

    #Add structured descriptions
    hf_dataset = add_struct_description(hf_dataset, ontology_dict, batch_size=batch_size)

    # Verify all examples have descriptions
    print(f"Examples with struct_desc: {sum(1 for ex in hf_dataset if 'struct_desc' in ex and ex['struct_desc'])}")

    print("Saving final dataset...")
    #Save the final dataset
    final_save_path = os.path.join(output_dir, dataset_name + "_with_descriptions")
    hf_dataset.save_to_disk(final_save_path)
    print(f"Final dataset saved to {final_save_path}")
    print(f"Final dataset size: {len(hf_dataset)} examples")

    return hf_dataset

In [48]:
hf_dataset = process_dataset()

Tokenizing dataset...
Loading tokenized dataset...


/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


Dataset info: Dataset({
    features: ['input_ids', 'soma_joinid', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'disease', 'disease_ontology_term_id', 'sex', 'tissue_general', 'tissue_general_ontology_term_id', 'length'],
    num_rows: 99999
})
Columns: ['input_ids', 'soma_joinid', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'disease', 'disease_ontology_term_id', 'sex', 'tissue_general', 'tissue_general_ontology_term_id', 'length']
Shape: (99999, 11)
Total number of examples: 99999
Adding structured descriptions...
Adding structured descriptions to all examples...
Examples with struct_desc: 99999
Saving final dataset...
Final dataset saved to /content/drive/MyDrive/cell2text_dataset_final/final_dataset_with_descriptions
Final dataset size: 99999 examples
